In [ ]:
import pandas as pd
import numpy as np
import sklearn
import requests

In [ ]:
import torch
import json, requests

from transformers import AutoModel, AutoModelForMaskedLM, AutoTokenizer, DataCollatorForLanguageModeling, TrainingArguments, Trainer

from datasets import Dataset
from peft import LoraConfig, get_peft_model

# device = "cuda" if torch.cuda.is_available() else "cpu"
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

hf_cache = "/Users/sguirale/tmp"

## Get Training Data

In [ ]:
raw_df = pd.read_csv("../Input/input.csv")

raw_df.drop(columns=["ID", "affinity_Kd_nM"], inplace=True)

## Take random 10k sample
raw_df = raw_df.sample(n=10000, random_state=1337)

raw_df.head()

In [ ]:
## Other Filters

raw_df = raw_df[raw_df['neg_log10_affinity_M'] >= 4]


print(f"Filtered dataset size: {len(raw_df)}")

## Load Enzyme Tokenizer


In [ ]:
## Load Enzyme Models
base_model_path = "Synthyra/ESM2-8M"

esm2_model = AutoModel.from_pretrained(
    base_model_path,
    trust_remote_code=True,
    attn_implementation="sdpa",
    cache_dir=hf_cache
).to(device).eval()


esm2_tokenizer = AutoTokenizer.from_pretrained(
    base_model_path,
    trust_remote_code=True,
    cache_dir=hf_cache
)

In [ ]:
## Function to get sequence embeddings
@torch.inference_mode()
def get_sequence_embeddings(model, sequences, batch_size=4):
    embedding_batches = []

    for start in range(0, len(sequences), batch_size):
        sequence_batch = sequences[start:start + batch_size]
        batch = esm2_tokenizer(
            sequence_batch,
            padding=True,
            truncation=True,
            return_tensors="pt",
        ).to(device)

        outputs = model(**batch)
        hidden = outputs.last_hidden_state

        ## Ignore padding during mean pooling.
        mask = batch["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)
        embedding_batches.append(pooled.cpu())

        del batch, outputs, hidden, mask, pooled
        if device == "cuda":
            torch.cuda.empty_cache()

    return torch.cat(embedding_batches, dim=0)

In [ ]:
aa_sequences = raw_df['seq'].tolist()

In [ ]:
aa_emb = get_sequence_embeddings(esm2_model, aa_sequences)

## Get Ligand Tokenizer

In [ ]:
ligand_model_name = "DeepChem/ChemBERTa-77M-MLM"

ligand_tokenizer = AutoTokenizer.from_pretrained(ligand_model_name, cache_dir=hf_cache)
ligand_model = AutoModel.from_pretrained(ligand_model_name, cache_dir=hf_cache).to(device).eval()

In [ ]:
## Function to get SMILES embeddings
@torch.inference_mode()
def get_smiles_embeddings(model, smiles, batch_size=32):
    embedding_batches = []

    for start in range(0, len(smiles), batch_size):
        smiles_batch = smiles[start:start + batch_size]
        batch = ligand_tokenizer(
            smiles_batch,
            padding=True,
            truncation=True,
            return_tensors="pt",
        ).to(device)

        outputs = model(**batch)
        hidden = outputs.last_hidden_state

        ## Ignore padding during mean pooling.
        mask = batch["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)
        embedding_batches.append(pooled.cpu())

        del batch, outputs, hidden, mask, pooled
        if device == "cuda":
            torch.cuda.empty_cache()

    return torch.cat(embedding_batches, dim=0)

In [ ]:
smiles_sequences = raw_df['smiles'].tolist()

In [ ]:
lig_emb = get_smiles_embeddings(ligand_model, smiles_sequences)

## Train Model

In [ ]:
## Make dataset of embeddings and neg_log10_affinity_M values
ligand_features = lig_emb.numpy()
enzyme_features = aa_emb.numpy()

if len(ligand_features) != len(enzyme_features) or len(ligand_features) != len(raw_df):
    raise ValueError("Ligand embeddings, enzyme embeddings, and targets must have the same number of rows")

## One row per enzyme-ligand pair, with both embeddings as numeric features.
X = np.concatenate([ligand_features, enzyme_features], axis=1)
y = pd.to_numeric(raw_df['neg_log10_affinity_M'], errors='coerce').to_numpy(dtype=np.float32)

valid_rows = np.isfinite(y)
X = X[valid_rows]
y = y[valid_rows]

print(f"Features: {X.shape}; targets: {y.shape}")

In [ ]:
## Inspect target and embedding coverage
print(raw_df['neg_log10_affinity_M'].describe())
print(f"Unique enzyme sequences: {raw_df['seq'].nunique()}")
print(f"Unique ligands: {raw_df['smiles'].nunique()}")
print(f"Duplicate enzyme-ligand pairs: {raw_df.duplicated(['seq', 'smiles']).sum()}")
print(f"NaN or infinite features: {np.isnan(X).sum() + np.isinf(X).sum()}")

In [ ]:
## Turnover is usually strongly right-skewed; fit the model in log space.
y_log = np.log1p(y)
# y_log = np.log10(y+1e-8)  # Add a small constant to avoid log(0)

In [ ]:
## Make train/validation/test splits
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    # y,
    y_log,
    test_size=0.2,
    random_state=1337
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train,
    y_train,
    test_size=0.25,
    random_state=1337
)


print(f"Train: {len(y_train)}, validation: {len(y_valid)}, test: {len(y_test)}")

In [ ]:
## Train Turnover Model
import xgboost as xgb

model = xgb.XGBRegressor(
    n_estimators=3000,
    max_depth=25,
    learning_rate=0.03,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    eval_metric='rmse',
    early_stopping_rounds=50,
    random_state=1337
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=False
)

In [ ]:
## Evaluate Turnover Model
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr

predictions_log = model.predict(X_test)

## For log1p
predictions = np.maximum(np.expm1(predictions_log), 0)
y_test_original = np.expm1(y_test)
y_train_original = np.expm1(y_train)

## For log10
# predictions = np.maximum(np.power(10, predictions_log) - 1e-8, 0)
# y_test_original = np.power(10, y_test) - 1e-8
# y_train_original = np.power(10, y_train) - 1e-8

## Evaluate in log space, the scale used for training.
log_rmse = np.sqrt(mean_squared_error(y_test, predictions_log))
log_mae = mean_absolute_error(y_test, predictions_log)
log_r2 = r2_score(y_test, predictions_log)

## Evaluate in original turnover units for interpretability.
rmse = np.sqrt(mean_squared_error(y_test_original, predictions))
mae = mean_absolute_error(y_test_original, predictions)
r2 = r2_score(y_test_original, predictions)

## Spearman correlation
spearman_corr, spearman_p = spearmanr(y_test_original, predictions)

## Median baselines on the matching target scales.
log_baseline = np.full_like(y_test, np.median(y_train))
original_baseline = np.full_like(y_test_original, np.median(y_train_original))

log_baseline_rmse = np.sqrt(mean_squared_error(y_test, log_baseline))
baseline_rmse = np.sqrt(mean_squared_error(y_test_original, original_baseline))
baseline_mae = mean_absolute_error(y_test_original, original_baseline)

print(f"Log RMSE: {log_rmse:.4f} (median baseline: {log_baseline_rmse:.4f})")
print(f"Log MAE:  {log_mae:.4f}")
print(f"Log R^2:  {log_r2:.4f}")
print(f"Original RMSE: {rmse:.4f} (median baseline: {baseline_rmse:.4f})")
print(f"Original MAE:  {mae:.4f} (median baseline: {baseline_mae:.4f})")
print(f"Spearman correlation: {spearman_corr:.4f} (p-value: {spearman_p:.4f})")
print(f"Original R^2:  {r2:.4f}")
print(f"Best iteration: {model.best_iteration}")

In [ ]:
## Compare with a regularized linear model
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

ridge_model = make_pipeline(
    StandardScaler(),
    Ridge(alpha=10.0)
)
ridge_model.fit(X_train, y_train)
ridge_predictions_log = ridge_model.predict(X_test)
ridge_predictions = np.maximum(np.expm1(ridge_predictions_log), 0)

ridge_log_rmse = np.sqrt(mean_squared_error(y_test, ridge_predictions_log))
ridge_log_mae = mean_absolute_error(y_test, ridge_predictions_log)
ridge_log_r2 = r2_score(y_test, ridge_predictions_log)

ridge_rmse = np.sqrt(mean_squared_error(y_test_original, ridge_predictions))
ridge_mae = mean_absolute_error(y_test_original, ridge_predictions)
ridge_r2 = r2_score(y_test_original, ridge_predictions)

print(f"Ridge log RMSE: {ridge_log_rmse:.4f}")
print(f"Ridge log MAE:  {ridge_log_mae:.4f}")
print(f"Ridge log R^2:  {ridge_log_r2:.4f}")
print(f"Ridge original RMSE: {ridge_rmse:.4f}")
print(f"Ridge original MAE:  {ridge_mae:.4f}")
print(f"Ridge original R^2:  {ridge_r2:.4f}")